# CCTV Commercial II-01: pilot
Private CCTV processing. No known-speed reference or prior Kia/Mercedes correction is used.

In [ ]:
from pathlib import Path
import hashlib,json,subprocess,sys
sources={'cctv_inference.py': '"""Sequential, speed-blind measurements for long static CCTV recordings.\n\nEvery source frame is decoded; detection and cuboids are sampled independently.\nOnly cars with recent image movement contribute cuboids. No reference speeds,\nMercedes/Kia learned scales, or expected-speed clipping are used.\n"""\nimport argparse\nfrom collections import defaultdict, deque\nimport gzip\nimport hashlib\nimport json\nfrom pathlib import Path\nimport time\nimport urllib.request\n\nimport cv2\nimport numpy as np\nfrom scipy.optimize import linear_sum_assignment\nfrom ultralytics import YOLO\n\nfrom validation_inference import MODEL_REPO, MODEL_REVISION, MODEL_FILE, MODEL_SHA, overlaps\n\n\ndef region_for(point, profile, width, height):\n    for name, polygon in profile[\'regions\'].items():\n        contour = np.asarray(polygon, np.float32) * [width, height]\n        if cv2.pointPolygonTest(contour.astype(np.float32), tuple(map(float, point)), False) >= 0:\n            return name\n    return None\n\n\ndef inference(video, output, camera, duration=None, resume_s=0, device=\'0\'):\n    config = json.loads((Path(__file__).parent/\'cctv_camera_profiles.json\').read_text())\n    profile = config[\'cameras\'][camera]\n    geometry_hz=profile.get(\'geometry_hz\',config[\'geometry_hz\'])\n    motion_minimum_span=profile.get(\'motion_history_minimum_span_s\',.4)\n    video = Path(video); output = Path(output); output.mkdir(parents=True, exist_ok=True)\n    with video.open(\'rb\') as stream:\n        source_sha = hashlib.file_digest(stream, \'sha256\').hexdigest()\n    assert source_sha == profile[\'sha256\'], \'Unexpected source recording\'\n    cap = cv2.VideoCapture(str(video)); assert cap.isOpened()\n    width, height = [int(cap.get(key)) for key in [cv2.CAP_PROP_FRAME_WIDTH, cv2.CAP_PROP_FRAME_HEIGHT]]\n    fps = float(cap.get(cv2.CAP_PROP_FPS)); total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))\n    stop_frame = total if duration is None else min(total, int(round(duration*fps)))\n    resume_frame = int(round(resume_s*fps))\n    models = output.parent/\'models\'; models.mkdir(exist_ok=True)\n    weights = models/\'urban_geometry.pt\'\n    if not weights.exists():\n        urllib.request.urlretrieve(f\'https://huggingface.co/{MODEL_REPO}/resolve/{MODEL_REVISION}/{MODEL_FILE}\', weights)\n    with weights.open(\'rb\') as stream:\n        assert hashlib.file_digest(stream, \'sha256\').hexdigest() == MODEL_SHA\n    tracker = YOLO(str(models/\'yolo11s.pt\'))\n    geometry = YOLO(str(weights))\n    tracker_config = output/\'bytetrack_10fps.yaml\'\n    tracker_config.write_text(\'tracker_type: bytetrack\\ntrack_high_thresh: 0.25\\ntrack_low_thresh: 0.10\\nnew_track_thresh: 0.25\\ntrack_buffer: 10\\nmatch_thresh: 0.80\\nfuse_score: True\\n\')\n    track_file = gzip.open(output/\'tracks.jsonl.gz\', \'wt\', encoding=\'utf-8\')\n    geom_file = gzip.open(output/\'observations.jsonl.gz\', \'wt\', encoding=\'utf-8\')\n    histories = defaultdict(deque)\n    counts = defaultdict(int)\n    next_detection, next_geometry = resume_frame, resume_s\n    start = time.monotonic(); index = 0; previous_pts = -1; timestamp_fallbacks = 0\n    checkpoints = [];ground_votes=defaultdict(list)\n    try:\n        while index < stop_frame:\n            ok = cap.grab()\n            if not ok:\n                break\n            pts = float(cap.get(cv2.CAP_PROP_POS_MSEC))/1000\n            expected = index/fps\n            timestamp = pts if np.isfinite(pts) and pts > previous_pts and abs(pts-expected) < .15 else expected\n            timestamp_fallbacks += int(timestamp != pts)\n            previous_pts = timestamp\n            if index < resume_frame or index + 1e-6 < next_detection:\n                index += 1; continue\n            ok, frame = cap.retrieve(); assert ok, f\'Cannot retrieve frame {index}\'\n            while next_detection <= index + 1e-6:\n                next_detection += fps/config[\'detection_hz\']\n            if counts[\'detection_frames\'] == 0:\n                cv2.imwrite(str(output/\'reference.jpg\'), frame)\n            detected = tracker.track(frame, persist=True, tracker=str(tracker_config), device=device,\n                                     imgsz=1280, classes=[2,3,5,7], conf=.10, half=True, verbose=False)[0]\n            boxes = detected.boxes.xyxy.cpu().numpy()\n            ids = detected.boxes.id.cpu().numpy().astype(int) if detected.boxes.id is not None else np.full(len(boxes), -1)\n            scores = detected.boxes.conf.cpu().numpy(); classes = detected.boxes.cls.cpu().numpy().astype(int)\n            # A reset at the resume boundary cannot silently join unrelated track IDs.\n            ids = np.where(ids >= 0, ids + (1000000 if resume_frame else 0), -1)\n            eligible = []\n            for k, (box, tid, score, cls) in enumerate(zip(boxes, ids, scores, classes)):\n                if tid < 0:\n                    continue\n                center = (box[:2] + box[2:])/2\n                region = region_for((center[0], box[3]), profile, width, height)\n                history = histories[int(tid)]\n                history.append((timestamp, center))\n                while history and timestamp-history[0][0] > 2.5:\n                    history.popleft()\n                displacement = float(np.linalg.norm(center-history[0][1]))\n                moving = timestamp-history[0][0] >= motion_minimum_span-.001 and displacement >= max(8, .10*(box[3]-box[1]))\n                row = dict(frame_index=index, time_s=timestamp, track_id=int(tid), class_id=int(cls),\n                           box=box.tolist(), confidence=float(score), region=region, moving=bool(moving))\n                track_file.write(json.dumps(row, separators=(\',\', \':\'))+\'\\n\'); counts[\'track_rows\'] += 1\n                if cls == 2 and moving and region and box[3]-box[1] >= 40:\n                    eligible.append(k)\n            counts[\'detection_frames\'] += 1\n            if timestamp >= next_geometry:\n                next_geometry = timestamp+1/geometry_hz-.01\n                if eligible:\n                    result = geometry.predict(frame, device=device, imgsz=1920, classes=[0], conf=.15,\n                                              half=True, verbose=False)[0]\n                    counts[\'geometry_frames\'] += 1\n                    if result.keypoints is not None and len(result.boxes):\n                        gb = result.boxes.xyxy.cpu().numpy(); gs = result.boxes.conf.cpu().numpy()\n                        kp = result.keypoints.data.cpu().numpy()\n                        overlap = overlaps(boxes[eligible], gb); ii, jj = linear_sum_assignment(1-overlap)\n                        for i,j in zip(ii,jj):\n                            if overlap[i,j] < .35:\n                                continue\n                            k = eligible[i]\n                            row = dict(frame_index=index, time_s=timestamp, track_id=int(ids[k]),\n                                       baseline_confidence=float(scores[k]), geometry_confidence=float(gs[j]),\n                                       association_iou=float(overlap[i,j]), baseline_box=boxes[k].tolist(),\n                                       geometry_box=gb[j].tolist(), keypoints=kp[j].tolist())\n                            geom_file.write(json.dumps(row, separators=(\',\', \':\'))+\'\\n\'); counts[\'observations\'] += 1\n                            ground_votes[int(ids[k])].append(bool(kp[j,:4,1].mean()>kp[j,4:,1].mean()))\n                        if len(checkpoints) < 16 and timestamp >= resume_s+len(checkpoints)*60:\n                            cv2.imwrite(str(output/f\'geometry_{index:06d}.jpg\'), result.plot())\n                            checkpoints.append(timestamp)\n            if counts[\'detection_frames\'] % 600 == 0:\n                track_file.flush(); geom_file.flush()\n                print(camera, f\'{timestamp:.1f}s / {stop_frame/fps:.1f}s\', dict(counts),\n                      \'runtime\', round(time.monotonic()-start,1), flush=True)\n                # Histories only hold active/recent tracks, rather than an hour of parked IDs.\n                stale = [tid for tid,h in histories.items() if h and timestamp-h[-1][0] > 10]\n                for tid in stale:\n                    del histories[tid]\n            if index % int(fps*300) < fps/config[\'detection_hz\']:\n                cv2.imwrite(str(output/f\'view_{int(timestamp):05d}.jpg\'), frame)\n            index += 1\n    finally:\n        cap.release(); track_file.close(); geom_file.close()\n    ground_fraction=float(np.mean([np.mean(v)>=.5 for v in ground_votes.values()])) if ground_votes else .5\n    ground_indices=list(range(4)) if ground_fraction>=.5 else list(range(4,8))\n    report = dict(camera_id=camera, source_video=video.name, source_sha256=source_sha,\n                  width=width, height=height, fps=fps, frames_reported=total,\n                  frames_decoded=index, expected_decoded_frames=stop_frame,\n                  decode_complete=index==stop_frame, start_s=resume_s, stop_s=stop_frame/fps,\n                  timestamps=\'Monotonic container PTS checked against frame/FPS; fallback when inconsistent\',\n                  timestamp_fallback_frames=timestamp_fallbacks, detection_hz=config[\'detection_hz\'],\n                  geometry_hz=geometry_hz, ground_indices=ground_indices,\n                  ground_order_vote_agreement=max(ground_fraction,1-ground_fraction),\n                  model_sha256=MODEL_SHA, model_revision=MODEL_REVISION, model_file=MODEL_FILE,\n                  counts=dict(counts), runtime_s=time.monotonic()-start,\n                  known_speeds_used=False, benchmark_vehicle_scales_used=False,\n                  id_reset_at_resume_s=resume_s if resume_s else None)\n    (output/\'inference_report.json\').write_text(json.dumps(report, indent=2)+\'\\n\', encoding=\'utf-8\')\n    assert report[\'decode_complete\'], f\'Source decode ended early: {index}/{stop_frame}\'\n    print(\'COMPLETE\', camera, dict(counts), round(report[\'runtime_s\'],1), flush=True)\n\n\nif __name__ == \'__main__\':\n    p=argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--video\',required=True);p.add_argument(\'--output\',required=True)\n    p.add_argument(\'--camera\',required=True);p.add_argument(\'--duration\',type=float)\n    p.add_argument(\'--resume-s\',type=float,default=0);p.add_argument(\'--device\',default=\'0\')\n    a=p.parse_args();inference(a.video,a.output,a.camera,a.duration,a.resume_s,a.device)\n', 'cctv_camera_profiles.json': '{\n  "schema_version": 1,\n  "description": "Manually selected road regions from each supplied camera view. Metric calibration is inferred from moving cars, without known speeds or road measurements.",\n  "detection_hz": 10,\n  "geometry_hz": 2.5,\n  "calibration_windows_s": [300, 600, 900],\n  "minimum_cars_per_region": 6,\n  "preferred_cars_per_region": 12,\n  "dimension_prior_m": {"length": 4.5, "width": 1.8},\n  "cameras": {\n    "commercial_ii_01": {\n      "display_name": "Commercial II-01",\n      "video": "NVR_ch6_main_20261009170002_20261009180000.mp4",\n      "sha256": "8e4437807fcae7c56f69e6cd55f04374d71e708630afbdfc811b1f192e10beee",\n      "regions": {\n        "left_carriageway": [[0,0.85],[0.30,0.52],[0.43,0.41],[0.44,0.53],[0.43,0.68],[0.56,0.715],[0.57,1],[0,1]],\n        "right_carriageway": [[0.57,0.415],[0.65,0.43],[0.75,0.52],[0.98,0.78],[1,0.92],[1,1],[0.57,1],[0.57,0.715],[0.56,0.60],[0.535,0.45]]\n      },\n      "limitations": "Downhill surface approximated by a separate local plane per carriageway. Parking and the median are excluded by region and motion filters."\n    },\n    "k_block_st20": {\n      "display_name": "K Block St 20 opposite side",\n      "geometry_hz": 10,\n      "motion_history_minimum_span_s": 0.1,\n      "calibration_minimum_observations": 4,\n      "calibration_minimum_span_s": 0.30,\n      "speed_window_s": 1.2,\n      "speed_minimum_span_s": 0.35,\n      "speed_quadratic_minimum_span_s": 0.8,\n      "video": "D02_20261009160351.mp4",\n      "sha256": "bc71067c55f50e21a39da6037617f418282fde1be2bb4d2fe47c6f52525fd5d2",\n      "regions": {\n        "near_approach": [[0,0.55],[0.14,0.29],[0.59,0.25],[0.63,0.32],[0.83,0.34],[0.85,0.84],[0.72,1],[0,1]],\n        "cross_street": [[0,0.40],[0.16,0.25],[0.25,0.19],[0.61,0.11],[0.75,0.10],[0.75,0.18],[0.59,0.25],[0.14,0.29],[0,0.55]]\n      },\n      "limitations": "Junction and near approach calibrated separately. Speeds are local 2D trajectory derivatives, allowing turns and changes in speed."\n    }\n  }\n}\n', 'validation_inference.py': '"""Speed-blind detection and cuboid extraction for constant-speed validation clips."""\nfrom pathlib import Path\nimport argparse\nimport hashlib\nimport json\nimport time\nimport urllib.request\n\nimport cv2\nimport numpy as np\nfrom scipy.optimize import linear_sum_assignment\nfrom ultralytics import YOLO\n\nMODEL_REPO=\'mehmetkeremturkcan/UrbanOmniDetect\'\nMODEL_REVISION=\'2701a2eec9a73afb5ffd852740d125dfcd065ce5\'\nMODEL_FILE=\'checkpoints/urbanomnidetect_yolo11x-p2_1920.pt\'\nMODEL_SHA=\'9aa55280a5ef1c9d89c0acb5baeb974b90fbdf3c1deead1413b24d7af249226d\'\n\n\ndef overlaps(a,b):\n    lo=np.maximum(a[:,None,:2],b[None,:,:2]);hi=np.minimum(a[:,None,2:],b[None,:,2:])\n    intersection=np.prod(np.maximum(0,hi-lo),axis=2)\n    return intersection/(np.prod(a[:,2:]-a[:,:2],axis=1)[:,None]+\n                         np.prod(b[:,2:]-b[:,:2],axis=1)[None,:]-intersection+1e-9)\n\n\ndef infer(folder,output,device=\'0\'):\n    output=Path(output);output.mkdir(parents=True,exist_ok=True)\n    model_dir=output.parent/\'models\';model_dir.mkdir(exist_ok=True)\n    weights=model_dir/\'urban_geometry.pt\'\n    if not weights.exists():\n        urllib.request.urlretrieve(f\'https://huggingface.co/{MODEL_REPO}/resolve/{MODEL_REVISION}/{MODEL_FILE}\',weights)\n    assert hashlib.sha256(weights.read_bytes()).hexdigest()==MODEL_SHA\n    geometry_model=YOLO(str(weights))\n    for video in sorted(Path(folder).rglob(\'*.MP4\')):\n        target=output/video.stem;target.mkdir(exist_ok=True)\n        tracker=YOLO(str(model_dir/\'yolo11s.pt\'))\n        cap=cv2.VideoCapture(str(video));assert cap.isOpened()\n        width=int(cap.get(cv2.CAP_PROP_FRAME_WIDTH));height=int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))\n        fps=float(cap.get(cv2.CAP_PROP_FPS));total=int(cap.get(cv2.CAP_PROP_FRAME_COUNT))\n        observations=[];tracks=[];times=[];start=time.monotonic();index=0\n        while True:\n            ok,frame=cap.read()\n            if not ok:break\n            pts=float(cap.get(cv2.CAP_PROP_POS_MSEC))/1000\n            timestamp=pts if abs(pts-index/fps)<.1 else index/fps\n            times.append(timestamp)\n            if index==0:cv2.imwrite(str(target/\'reference.jpg\'),frame)\n            detected=tracker.track(frame,persist=True,tracker=\'bytetrack.yaml\',device=device,\n                                   imgsz=1280,classes=[2,5,7],conf=.15,verbose=False)[0]\n            boxes=detected.boxes.xyxy.cpu().numpy()\n            ids=detected.boxes.id.cpu().numpy().astype(int) if detected.boxes.id is not None else np.full(len(boxes),-1)\n            scores=detected.boxes.conf.cpu().numpy()\n            for box,tid,score in zip(boxes,ids,scores):\n                tracks.append({\'frame_index\':index,\'time_s\':timestamp,\'track_id\':int(tid),\n                               \'box\':box.tolist(),\'confidence\':float(score)})\n            selected=[i for i,box in enumerate(boxes) if box[3]-box[1]>=45 and ids[i]>=0]\n            if selected:\n                result=geometry_model.predict(frame,device=device,imgsz=1920,classes=[0],conf=.15,verbose=False)[0]\n                if result.keypoints is not None and len(result.boxes):\n                    gb=result.boxes.xyxy.cpu().numpy();gs=result.boxes.conf.cpu().numpy()\n                    kp=result.keypoints.data.cpu().numpy()\n                    overlap=overlaps(boxes[selected],gb);ii,jj=linear_sum_assignment(1-overlap)\n                    for i,j in zip(ii,jj):\n                        if overlap[i,j]<.25:continue\n                        k=selected[i]\n                        observations.append({\'frame_index\':index,\'time_s\':timestamp,\'track_id\':int(ids[k]),\n                            \'baseline_confidence\':float(scores[k]),\'geometry_confidence\':float(gs[j]),\n                            \'association_iou\':float(overlap[i,j]),\'baseline_box\':boxes[k].tolist(),\n                            \'geometry_box\':gb[j].tolist(),\'keypoints\':kp[j].tolist()})\n                    if len(observations)%20<2:\n                        cv2.imwrite(str(target/f\'geometry_{index:04d}.jpg\'),result.plot())\n            index+=1\n        cap.release()\n        assert index==total\n        (target/\'observations.json\').write_text(json.dumps(observations),encoding=\'utf-8\')\n        (target/\'tracks.json\').write_text(json.dumps(tracks),encoding=\'utf-8\')\n        if observations:\n            points=np.array([r[\'keypoints\'] for r in observations])\n            vote=points[:,:4,1].mean(axis=1)>points[:,4:,1].mean(axis=1)\n            ground_indices=list(range(4)) if vote.mean()>=.5 else list(range(4,8))\n            agreement=float(max(vote.mean(),1-vote.mean()))\n        else:ground_indices=[];agreement=0.0\n        report={\'source_video\':video.name,\'width\':width,\'height\':height,\'fps\':fps,\'frames\':total,\n                \'duration_s\':total/fps,\'timestamps\':times,\'model_sha256\':MODEL_SHA,\n                \'model_revision\':MODEL_REVISION,\'model_filename\':MODEL_FILE,\'ground_indices\':ground_indices,\n                \'ground_order_vote_agreement\':agreement,\'runtime_s\':time.monotonic()-start,\n                \'observations\':len(observations),\'reference_speeds_used_by_inference\':False}\n        (target/\'geometry_run_report.json\').write_text(json.dumps(report,indent=2),encoding=\'utf-8\')\n        print(video.name,\'frames\',index,\'geometry\',len(observations),\'seconds\',round(report[\'runtime_s\'],1),flush=True)\n\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--input\',required=True);parser.add_argument(\'--output\',required=True)\n    parser.add_argument(\'--device\',default=\'0\')\n    args=parser.parse_args();infer(args.input,args.output,args.device)\n'}
camera='commercial_ii_01'
video_name='NVR_ch6_main_20261009170002_20261009180000.mp4'
root=Path('/kaggle/working/pipeline');root.mkdir(exist_ok=True)
for name,source in sources.items():(root/name).write_text(source,encoding='utf-8')
subprocess.check_call([sys.executable,'-m','pip','install','-q','ultralytics==8.4.175','lap==0.5.13'])
videos=list(Path('/kaggle/input').rglob(video_name));assert len(videos)==1,(video_name,videos)
out=Path('/kaggle/working/cctv_outputs')/camera;out.mkdir(parents=True,exist_ok=True)
def run(script,*args):subprocess.check_call([sys.executable,str(root/script),*map(str,args)])
run('cctv_inference.py','--video',videos[0],'--output',out/'pilot','--camera',camera,'--duration',900)

import torch,cv2,ultralytics
report={'camera_id':camera,'torch':torch.__version__,'opencv':cv2.__version__,
        'ultralytics':ultralytics.__version__,'gpu':torch.cuda.get_device_name(),
        'source_sha256':{name:hashlib.sha256((root/name).read_bytes()).hexdigest() for name in sources}}
(out/'runtime.json').write_text(json.dumps(report,indent=2),encoding='utf-8')
print('Job complete:',camera)
